In [ ]:
# all the required packages:
! conda list 

In [ ]:
# select first the conda env "image-analysis"
# which includes the following packages:
# downloading following packages: 
# conda install -c conda-forge czifile
# conda install -c anaconda scikit-image
# conda install -c anaconda pathlib
# conda install -c conda-forge microfilm


# importing the required packages
import czifile # to import a .czi file
# from PIL import Image # to convert .czi file to a .tif file
import skimage # general package for manipulating imaging data
from pathlib import Path # for file path 
import numpy as np
import matplotlib.pyplot as plt
from microfilm.microplot import microshow # for viewing multichannel image data
from skimage.transform import rotate # to rotate the image as a control
from skimage.restoration import rolling_ball # for image processing
from skimage.filters import gaussian # for image processing
from skimage.feature import peak_local_max # for local max detection
# import pyclesperanto_prototype as cle # package for additional vizualization


In [ ]:
# importing a czi file, checking the shape of the image array and getting info about image acquisition
path = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SLM.czi"
image = czifile.imread(path)
image.shape

In [ ]:
# this retrieves the metadata and the extracts to pixel_to_um conversion

from lxml import etree # required library to load the metadata

czi = czifile.CziFile("/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SLM.czi")
czi_xml_str = czi.metadata() # gets the metadata in a xml string format
czi_parsed = etree.fromstring(czi_xml_str) # parses the czi_xml_str file

# finds the strings 
size_x = czi_parsed.find(".//SizeX")
size_y = czi_parsed.find(".//SizeY")
scaling_x = czi_parsed.find(".//ScalingX")
scaling_y = czi_parsed.find(".//ScalingY")

# Extracting the required values to calculate the 
size_x_value = int(size_x.text)
size_y_value = int(size_y.text)
scaling_x_value = float(scaling_x.text)
scaling_y_value = float(scaling_y.text)

# calculater the pixel to micrometer (um)
# first checking whether the X and Y dimensions of the image are equal
if size_x_value == size_y_value and scaling_x_value == scaling_y_value:
    pixel_size = ((scaling_x_value*1000000000)/size_x_value) # conversion from meter to micrometer
    
print(pixel_size, "um per pixel") 

In [ ]:
# getting rid of all the extra channels, splitting the channels and showing them one by one.
image_squeezed = np.squeeze(image)
image_squeezed.shape
vglut1 = image_squeezed[0,:,:]
psd95 = image_squeezed[1,:,:]

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(vglut1, ax=axs[0], label_text = 'VLGUT1')
microshow(psd95, ax=axs[1], label_text = 'PSD95')


In [ ]:
# now a preproccesing step, removal of background with rolling ball radius of 10x
background_vglut1 = rolling_ball(vglut1, radius = 10)
background_psd95 = rolling_ball(psd95, radius = 10)
vglut1_bs = vglut1 - background_vglut1
psd95_bs = psd95 - background_psd95

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(vglut1_bs, ax=axs[0], label_text = 'VLGUT1')
microshow(psd95_bs, ax=axs[1], label_text = 'PSD95')

In [ ]:
# for the local maxima detection, it is advised to perform a gaussian blur (in this case a light one with sigma 1)
psd95_pre = gaussian(psd95_bs, sigma=1, preserve_range=True)

vglut1_pre = gaussian(vglut1_bs, sigma=1, preserve_range=True)

fig, axs = plt.subplots(2, 2, figsize = (30, 30))
microshow(psd95_bs, ax=axs[0,0])
microshow(psd95_pre, ax=axs[0,1])
microshow(vglut1_bs, ax=axs[1,0])
microshow(vglut1_pre, ax=axs[1,1])

In [ ]:
def local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_threshold, psd95_threshold):
    
    """Detects the local intensity peak of each channel processed
    
        Args:
            vglut1_preprocessed (np.array): processed image of vlgut1, which is background substracted and has a gaussian blur
            psd95_preprocessed (np.array): processed image of psd95, which is background substracted and has a gaussian blur
            vglut1_threshold (float): thresholding of the vlgut1 image for local peak detection
            psd95_threshold (float): thresholding of the psd95 image for local peak detection
    
        Returns:
            vglut1_coord (np.array): coordinates of vglut1 local peak maxima
            psd95_coord (np.array): coordinates of psd95 local peak maxima
            psd95_rot_coord (np.array): coordinates of psd95 local peak maxima
            
            plot of vglut1, psd95, psd95_rot images with the local peak maxima overlaid
    """
    
    # Thresholding the image for local peak maximum detection
    vglut1_coord = peak_local_max(vglut1_preprocessed, min_distance = 1, threshold_abs=vglut1_threshold)
    psd95_coord = peak_local_max(psd95_preprocessed, min_distance = 1, threshold_abs=psd95_threshold)

    # Rotating an image (psd95) as a control
    psd95_rot = rotate(psd95_preprocessed, 90)
    psd95_rot_coord = peak_local_max(psd95_rot, min_distance= 1, threshold_abs = psd95_threshold)
    
    # Showing the local peaks with coordinates together with the images
    fig, axs = plt.subplots(1, 3, figsize=(30, 30))

    axs[0].imshow(vglut1_preprocessed, cmap='gray')
    axs[0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    axs[0].set_title('vglut1_pre')

    axs[1].imshow(psd95_preprocessed, cmap='gray')
    axs[1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    axs[1].set_title('psd95_pre')

    axs[2].imshow(psd95_rot, cmap='gray')
    axs[2].plot(psd95_rot_coord[:, 1], psd95_rot_coord[:, 0], 'm.')
    axs[2].set_title('psd95_rot')
    
    return vglut1_coord, psd95_coord, psd95_rot_coord

In [ ]:
vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(vglut1_pre, psd95_pre, 300, 300)
print(vglut1_coord.shape, psd95_coord.shape, psd95_rot_coord.shape)

In [ ]:

# This block of code calculates the unique colocalized VGLUT1-PSD95 count, meaning it only has unique pairs of colocalized spots.
# It checks of every VLGUT1 and PSD95 spot, whether there are any PSD95 (for VGLUT1) and VLGUT1 (for PSD95) spots in their vicinity within a specific maximum distance and counts it. 
# So it can have for one spot, multiple pairs, and thus including multi-synaptic boutons.

from scipy.spatial.distance import cdist

def count_coloc_spots(vglut1_coordinates, psd95_coordinates, pixel_size_um, max_distance_um):
    
    """Counts the number of colocalized pre and postsynaptic spots based on the coordinates array of the local maxima detection
    
    Args:
        vlgut1_coordinates (np.array): coordinates of the local peak maxima in the vlgut1 channel
        psd95_coordinates (np.array): coordinates of the local peak maxima in the psd95 channel
        pixel_size_um (float): size of a pixel in um, based on image settings
        max_distance_um (float): value of the maximum colocalization distance between the spots of each channel in um
    
    Returns:
        colocalized spot count
    """
    
    # calculate the max distance in pixels with the pixel_size_um and max_distance_um
    max_distance_px = max_distance_um/pixel_size_um
    
    # Calculate pairwise distances between spots in vlgut1 and psd95 channel
    distances_vglut1_to_psd95 = cdist(vglut1_coordinates, psd95_coordinates)
    distances_psd95_to_vglut1 = cdist(psd95_coordinates, vglut1_coordinates)

    # Find unique colocalized spots
    colocalized_spots = set()

    # Iterate over distances from vglut1_coordinates to psd95_coordinates
    for i in range(len(vglut1_coordinates)):
        # Check if the current spot in vlgut1 has nearby spots in psd95
        colocalized_indices = [j for j, distance in enumerate(distances_vglut1_to_psd95[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((i, j))

    # Iterate over distances from psd95_coordinates to vglut1_coordinates
    for i in range(len(psd95_coordinates)):
        # Check if the current spot in Channel 2 has nearby spots in Channel 1
        colocalized_indices = [j for j, distance in enumerate(distances_psd95_to_vglut1[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((j, i))

    # Get the count of unique colocalized spots
    colocalized_spot_count = len(colocalized_spots)
    
    return colocalized_spot_count
    

In [ ]:
vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(vglut1_pre, psd95_pre, 500, 500)
colocalized_spot_count = count_coloc_spots(vglut1_coord, psd95_rot_coord, pixel_size, 0.2)
print(colocalized_spot_count)

In [ ]:
# writing a wrapper function for the two functions, which is easier to implement for the bayesian optimization
def comb_func(vglut1_preprocessed, psd95_preprocessed, vglut1_threshold, psd95_threshold, pixel_size_um, max_distance_um):
    
    # probing the first function to get the parameters as input for the second function
    vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_threshold, psd95_threshold)
    
    # probing the second function where psd95 is not rotated, the actual condition
    colocalized_spot_count = count_coloc_spots(vglut1_coord, psd95_coord, pixel_size_um, max_distance_um)
    # probing the second function where psd95 is rotated, the internal control condition
    colocalized_spot_count_rot = count_coloc_spots(vglut1_coord, psd95_rot_coord, pixel_size_um, max_distance_um)
    
    # getting the scale differences between the spot count of the normal situation and psd95 rotated
    scaled_spot_count_dif = (colocalized_spot_count - colocalized_spot_count_rot)/colocalized_spot_count
    
    return colocalized_spot_count, colocalized_spot_count_rot, scaled_spot_count_dif

In [ ]:
comb_func(vglut1_pre, psd95_pre, 628, 674, pixel_size, 0.1) # the region of max scaled_spot_count_dif

In [ ]:
from bayes_opt import BayesianOptimization

# setting the static variables
vglut1_preprocessed = vglut1_pre  # Assign vglut1_pre to vglut1_preprocessed
psd95_preprocessed = psd95_pre  # Assign psd95_pre to psd95_preprocessed
pixel_size_um = pixel_size  # Assign pixel_size to pixel_size_um


# define the objective function for optimization
def objective(vglut1_threshold, psd95_threshold, max_distance_um):
    colocalized_spot_count, colocalized_spot_count_rot, scaled_spot_count_dif = comb_func(
        vglut1_preprocessed,
        psd95_preprocessed,
        vglut1_threshold,
        psd95_threshold,
        pixel_size_um,
        max_distance_um
    )
    return scaled_spot_count_dif # Maximize the scaled_spot_count_dif

# Define the parameter bounds for optimization
param_bounds = {'vglut1_threshold': (500, 700), 
                'psd95_threshold': (500, 700), 
                'max_distance_um': (0.01, 1)} 

# Create an instance of the BayesianOptimization class
optimizer = BayesianOptimization(f=objective, 
                                 pbounds=param_bounds,
                                 verbose=2,
                                 random_state=1,)

# Perform Bayesian optimization, total of 20 iterations
optimizer.maximize(init_points=5, n_iter=15)

# Get the optimized parameters and the maximum scaled_spot_count_dif
optimal_params = optimizer.max['params']
max_scaled_spot_count_dif = optimizer.max['target']

# Print the results
print("Optimized Parameters:")
print("vlgut1_threshold:", optimal_params['vglut1_threshold'])
print("psd95_threshold:", optimal_params['psd95_threshold'])
print("max_distance:", optimal_params['max_distance_um'])
print("Maximized scaled_spot_count_dif:", max_scaled_spot_count_dif)


In [ ]:
# getting the data into a dataframe and writing out the data into a csv

import os
import pandas as pd

# get the filename
filename = os.path.splitext(os.path.basename(path))[0]
split_filename = filename.split("_")

# get only the experimental parameters from the filename
index_nums = [0, 1, 2, 3, 10, 11] # the indexes of the elements that I would like to extract
desired_parts = [split_filename[val] for val in index_nums]
desired_filename = "_".join(desired_parts)

# join the dictionary of the optimal parameters with the filename as rowname
df = pd.DataFrame.from_dict(optimal_params, orient = "index", columns = [desired_filename])

# adding the most important parameter, the max_scaled_spot_difference
df.loc["max_scaled_spot_difference"] = max_scaled_spot_count_dif

# pivoting the dataframe
dic = df.to_dict(orient='dict')
df_final = pd.DataFrame.from_dict(dic, orient='index')

# Writing the dataframe in a csv format into the a specific folder
output_filename = "opt_params_" + desired_filename + ".csv"
output_path = "/mnt/d/code/phd/image-analysis/synapse-counting/output_data/" + output_filename
df_final.to_csv(output_path)